In [ ]:
import lightkurve
from lightkurve import DesignMatrix, RegressionCorrector
import matplotlib.pyplot as plot
import numpy
%matplotlib inline
plot.style.use('seaborn-v0_8')

In [ ]:
star_id = [
    219777401,
    199710583,
    198358825,
    20352516,
    238229705,
    467113954,
    297701617,
    73248090,
    279322914,
    38850860,
    238229705,
    35655828,
    76903841,
    102625324,
    342556321,
    176871438,
    238229705
][16]

star_id = f'TIC {star_id}'

In [ ]:
search_result = lightkurve.search_targetpixelfile(star_id, mission = 'TESS', cadence = 'long')
search_result

In [ ]:
pixel_files = search_result[:10].download_all()

In [ ]:
collection = lightkurve.LightCurveCollection(None)
for pixel_file in pixel_files[:]:
    pixel_file = pixel_file[numpy.isfinite(pixel_file.flux).all(axis = (1, 2))]
    pixel_file = pixel_file[numpy.isfinite(pixel_file.flux_err).all(axis = (1, 2))]

    aperture_mask = pixel_file.pipeline_mask
    # aperture_mask = pixel_file.create_threshold_mask(threshold = 10, reference_pixel = 'center')
    uncorrected_lc = pixel_file.to_lightcurve(aperture_mask = aperture_mask)

    # design_matrix = DesignMatrix(pixel_file.flux[:, ~aperture_mask]).pca(5).append_constant()
    # lc = RegressionCorrector(uncorrected_lc).correct(design_matrix)
    lc = uncorrected_lc
    lc = lc.normalize()#.flatten(niters = 10).remove_outliers(10.0)
    lc.plot()
    collection.append(lc)

plot.show()

In [ ]:
lc = collection[:].stitch()
lc = lc#.flatten(break_tolerance = 10, niters = 10)#.remove_outliers(sigma = 10.0)
lc.plot()
plot.show()

In [ ]:
periods = numpy.logspace(0.5, 1.7, 10 ** 4, base = 10.0)
periodogram = lc.to_periodogram(method = 'boxleastsquares', period = periods, frequency_factor = 10 ** 6)
periodogram.plot()
plot.show()

In [ ]:
period = periodogram.period_at_max_power
transit_time = periodogram.transit_time_at_max_power
transit_duration = periodogram.duration_at_max_power
period, transit_time, transit_duration

In [ ]:
# transit_time += #period / 2 # 2nd eclipse
# period *= 2 # 2nd eclipse
# period /= 2
folded_lc = lc.fold(period = period, epoch_time = transit_time)
binned_lc = folded_lc.bin(time_bin_size = 0.01)
ax = folded_lc.scatter()
binned_lc.plot(ax = ax, c = 'r')
d = 0.5
# ax.set_xlim(-d, d)
# ax.set_xlim(-13, -10)
# ax.set_ylim(0.99, 1.01)
plot.show()
transit_depth = 1 - numpy.nanmin(binned_lc.flux.value)
transit_depth

In [ ]:
fig, axes = plot.subplots(2, sharex = True)
pixel_file_range = (0, 1)

c_lc = lightkurve.LightCurveCollection([pixel_file.to_lightcurve() for pixel_file in pixel_files[pixel_file_range[0]:pixel_file_range[1]]]).stitch()
centroids = numpy.concatenate([pixel_file.estimate_centroids() - numpy.nanmean(pixel_file.estimate_centroids(), axis = 1).reshape(-1, 1) for pixel_file in pixel_files[pixel_file_range[0]:pixel_file_range[1]]], axis = 1)

axes[0].scatter(c_lc.time.value, c_lc.flux.value, color = 'black', s = 1)
axes[0].set_ylabel('Flux')

axes[1].scatter(c_lc.time.value, centroids[0], color = 'blue', s = 1)
axes[1].set_ylabel('Row')

axes[1].scatter(c_lc.time.value, centroids[1], color = 'red', s = 1)
axes[1].set_ylabel('Column')
axes[1].set_xlabel('Time')

plot.show()